# 3. Off-target screening

Rank a whole protein set for one compound to look for off-targets. This is the analysis
behind Figure 5 of the paper, run here on 100 human proteins instead of the full
20,421-protein proteome.

The sample set contains the seven clinically validated clozapine targets, so the
ranking can be checked against pharmacology.

**Runtime** — about 2 seconds on a laptop CPU for 100 proteins.

## 1. Setup

The model runs on the CPU. `cfg.training.device` must be set **before** the model is built,
because the device is read at construction time.

In [ ]:
import os
import sys
import warnings

import pandas as pd
from omegaconf import OmegaConf

warnings.filterwarnings('ignore')

# Run from the repository root so that the relative paths below resolve.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

CHECKPOINT = 'checkpoints/application.pth'
assert os.path.exists(CHECKPOINT), (
    f'{CHECKPOINT} not found. Download the weights from Zenodo first '
    '(see checkpoints/README.md).')

cfg = OmegaConf.load('configs/default.yaml')
cfg.model.type = 'aa3di_gmf'   # must match the checkpoint
cfg.training.device = 'cpu'    # set before build_model/load_model
cfg.training.cuda = False

## 2. Load the model and the protein set

In [ ]:
from omnibind.predict import load_model

model = load_model(cfg, CHECKPOINT)

proteins = pd.read_csv('data/sample/sample_proteins.csv')
protein_data = {
    row['gene_name']: (row['aa_sequence'], row['3di_sequence'])
    for _, row in proteins.iterrows()
}
print(f'{len(protein_data)} proteins loaded')

## 3. Screen one compound

Clozapine is an antipsychotic that binds a broad range of receptors, and it is absent
from the training data.

In [ ]:
from omnibind.predict import batch_predict_proteins

CLOZAPINE = 'CN1CCN(CC1)C1=Nc2cc(Cl)ccc2Nc2ccccc12'

results = batch_predict_proteins(CLOZAPINE, protein_data, model, cfg)

failed = results['predicted_ki'].isna().sum()
print(f'predicted {len(results) - failed} / {len(results)} proteins')

## 4. Rank the proteins

The paper ranks by the mean over the four endpoints.

In [ ]:
results['mean_score'] = results[['predicted_ki', 'predicted_kd',
                                 'predicted_ic50', 'predicted_ec50']].mean(axis=1)
ranked = (results.dropna(subset=['mean_score'])
                 .sort_values('mean_score', ascending=False)
                 .reset_index(drop=True))
ranked.index += 1
ranked.head(10)[['protein_id', 'mean_score', 'predicted_ki', 'predicted_ic50']]

## 5. Compare with the known pharmacology

The seven experimentally validated clozapine targets are listed below. If the model
works, they should appear near the top of the ranking.

In [ ]:
KNOWN_TARGETS = ['HTR2A', 'HRH1', 'ADRA1A', 'ADRA1B', 'ADRA1D', 'CHRM1', 'HTR2C']

rank_of = {g: int(ranked.index[ranked['protein_id'] == g][0]) for g in KNOWN_TARGETS}
summary = (pd.DataFrame({'target': KNOWN_TARGETS,
                         'rank': [rank_of[g] for g in KNOWN_TARGETS]})
             .sort_values('rank').reset_index(drop=True))
print(f'{(summary["rank"] <= 10).sum()} of {len(summary)} known targets in the top 10')
summary

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['tab:red' if p in KNOWN_TARGETS else 'lightgray'
          for p in ranked['protein_id']]
ax.bar(range(1, len(ranked) + 1), ranked['mean_score'], color=colors, width=1.0)
ax.set_xlabel('rank')
ax.set_ylabel('mean predicted score')
ax.set_title('Clozapine: known targets (red) among 100 proteins')
fig.tight_layout()

## 6. Cumulative detection

How much of the protein set has to be searched to recover the known targets.

In [ ]:
hits = np.zeros(len(ranked) + 1)
for r in rank_of.values():
    hits[r:] += 1
detection = 100 * hits / len(KNOWN_TARGETS)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.step(range(len(detection)), detection, where='post', color='tab:red')
ax.set_xlabel('proteins searched')
ax.set_ylabel('known targets recovered (%)')
ax.set_ylim(0, 105)
fig.tight_layout()

## 7. Comparing two compounds

Clomipramine shares a tricyclic scaffold with clozapine but acts mainly on the
serotonin and norepinephrine transporters. A useful model should separate them.

In [ ]:
CLOMIPRAMINE = 'CN(C)CCCN1c2ccccc2CCc2ccc(Cl)cc21'

clomi = batch_predict_proteins(CLOMIPRAMINE, protein_data, model, cfg)
clomi['mean_score'] = clomi[['predicted_ki', 'predicted_kd',
                             'predicted_ic50', 'predicted_ec50']].mean(axis=1)
clomi_ranked = (clomi.dropna(subset=['mean_score'])
                     .sort_values('mean_score', ascending=False)
                     .reset_index(drop=True))
clomi_ranked.index += 1

pd.DataFrame({
    'clozapine': ranked['protein_id'].head(5).tolist(),
    'clomipramine': clomi_ranked['protein_id'].head(5).tolist(),
}, index=range(1, 6))

## 8. Using your own data

`batch_predict_proteins` takes a `{protein_id: (aa_sequence, 3di_sequence)}` dictionary,
so any protein set works as long as both sequences are present and equally long. See
`data/README.md` for generating 3Di sequences with ProstT5.

The script version reads the two sequence dictionaries from pickle files:

```bash
python scripts/offtarget_screening.py \
    test.checkpoint_path=./checkpoints/application.pth \
    test.fixed_smiles="<compound_smiles>" \
    test.seq_pkl_path=proteins_seq.pkl \
    test.ss_pkl_path=proteins_ss.pkl
```